# Train the transformers on a free Colab GPU

Fine-tunes DistilRoBERTa for every configuration in the v3 report. On a T4 the whole notebook takes
roughly 40-60 minutes; on my laptop CPU the same runs would take a day and a half.

1. **Runtime > Change runtime type > T4 GPU**
2. Run the cells in order. The second cell asks you to upload `colab_bundle.zip`
   (made locally with `python -m spam_detector.colab_bundle`).
3. The last cell downloads `runs.zip`. Unzip it into `artifacts/runs/` in the project.

Each run is skipped if its results already exist, so if Colab disconnects, reconnect,
re-upload the bundle and run all again.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import files
import os, zipfile
if not os.path.exists("src"):
    uploaded = files.upload()  # choose colab_bundle.zip
    zipfile.ZipFile(next(iter(uploaded))).extractall(".")
!pip -q install "transformers>=4.44,<5" ftfy
!pip -q install --no-deps -e .
import hashlib
print("corpus sha256:", hashlib.sha256(open("data/processed/corpus.parquet", "rb").read()).hexdigest()[:16])

## Runs

| Run | Trains on | Purpose |
|---|---|---|
| `sms-email_seed0..2` | SMS + email, train split | main results, 3 seeds for error bars |
| `sms_seed0` | SMS only | cross-channel transfer |
| `email_seed0` | email only | cross-channel transfer |
| `final_seed0` | train + test + future | the deployed model (saved) |

In [ ]:
RUNS = [
    ("sms-email_seed0", "--seed 0"),
    ("sms-email_seed1", "--seed 1"),
    ("sms-email_seed2", "--seed 2"),
    ("sms_seed0", "--domains sms --seed 0"),
    ("email_seed0", "--domains email --seed 0"),
    ("final_seed0", "--final --save-model --seed 0"),
]
for name, flags in RUNS:
    if os.path.exists(f"artifacts/runs/{name}/predictions.parquet"):
        print("skip", name)
        continue
    print("=" * 20, name)
    # get_ipython().system streams the training log into the notebook as it runs
    get_ipython().system(f"python -m spam_detector.models.transformer {flags}")
    assert os.path.exists(f"artifacts/runs/{name}/predictions.parquet"), f"{name} failed, see the log above"

In [ ]:
!cd artifacts && zip -qr ../runs.zip runs && cd .. && ls -lh runs.zip
files.download("runs.zip")